### 1 - Descrição
Neste trabalho, você deverá ser capaz de construir um pipeline de dados utilizando tecnologias na nuvem. O pipeline irá envolver a busca, coleta, modelagem, carga e análise dos dados.

### 2 - OBJETIVO
O objetivo deste trabalho é desenvolver um pipeline de dados utilizando a plataforma Databricks, com suporte das tecnologias Apache Spark e SQL, para processar, transformar e analisar dados relacionados a acidentes de trânsito. A proposta envolve a construção de uma arquitetura de dados eficiente e escalável, capaz de:

1 - Ingerir dados brutos de acidentes de trânsito, provenientes de arquivos CSV ou outras fontes estruturadas;

2 - Realizar o tratamento e transformação dos dados, como normalização de colunas (ex.: transformar nomes dos dias da semana em códigos), conversão de tipos e remoção de inconsistências;

3 - Criar tabelas intermediárias e dimensões (como dias) para facilitar análises posteriores, utilizando joins entre tabelas;

4 - Aplicar consultas SQL para gerar insights, como a distribuição de acidentes por dia da semana, tipo de ocorrência, horário, entre outros;

5- Otimizar o fluxo de dados com o uso de boas práticas de engenharia de dados no Spark, como particionamento, persistência em memória e uso eficiente de DataFrames;

6 - E em projetos futuros, gerar visualizações e relatórios dentro do ambiente Databricks, apoiando a tomada de decisão baseada em dados.

### 3 - COLETA DE DADOS
A etapa de coleta de dados deste projeto foi realizada a partir de fontes públicas e confiáveis, com o objetivo de reunir informações relevantes e atualizadas sobre acidentes de trânsito ocorridos em rodovias federais brasileiras, bem como dados de referência geográfica e populacional dos estados brasileiros. As fontes utilizadas foram:

- Base de Acidentes da Polícia Rodoviária Federal (PRF)
Os dados brutos sobre acidentes foram obtidos por meio do Portal de Dados Abertos da PRF, que disponibiliza registros detalhados dos acidentes de trânsito ocorridos em rodovias federais em todo o território nacional.
Esses dados incluem informações como: Data e horário do acidente, tipo de ocorrência, condições climáticas e localização, dentre outros.

- Arquivo com código dos estados brasileiros.


Os arquivos foram disponibilizados no formato .CSV, relativo apenas ao ano de 2024, e foram integrados ao ambiente Databricks para tratamento e análise.


### 4 - CATÁLOGO DE DADOS
![](https://raw.githubusercontent.com/MdotSouza/PUCRio/main/EngDados_MVP/resources/catalogo_registros.png)

![](https://raw.githubusercontent.com/MdotSouza/PUCRio/main/EngDados_MVP/resources/catalogo_estados.png)

### 5 - PIPELINE DE DADOS

RECURSOS COMUNS PARA TODAS AS CAMADAS 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
"""
:param caminho: Str
  endereço do arquivo csv para ser carregado
:param codificacao: Str
  formato de caracteres (encoding) que o arquivo csv utiliza
:return DataFrame:
"""
def buildDF(caminho, codificacao):
  return spark.read.format("csv") \
    .option("inferSchema", True) \
    .option("header", True) \
    .option("sep", ";") \
    .option("encoding", codificacao) \
    .load(caminho)

BRONZE LAYER

In [0]:
%sql
DROP DATABASE IF EXISTS datatran_bronze CASCADE;
CREATE DATABASE datatran_bronze;

In [0]:
PATH_ACIDENTES = "dbfs:/FileStore/resources/datatran2024.csv" 
ENC_ACIDENTES = "Windows-1250"
df_acidentes = buildDF(PATH_ACIDENTES, ENC_ACIDENTES)
df_acidentes.write.mode("overwrite").saveAsTable("datatran_bronze.acidentes")

In [0]:
PATH_ESTADOS = "dbfs:/FileStore/resources/estados.csv"
ENC_ESTADOS = "UTF-8"
df_estados = buildDF(PATH_ESTADOS, ENC_ESTADOS)
df_estados.write.mode("overwrite").saveAsTable("datatran_bronze.estados")

In [0]:
%sql
SELECT * FROM datatran_bronze.acidentes LIMIT 5;

id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
571789.0,2024-01-01,segunda-feira,2025-04-14T03:56:00.000+0000,ES,101,38,CONCEICAO DA BARRA,Ultrapassagem Indevida,Colisăo lateral sentido oposto,NA,Plena Noite,Crescente,Céu Claro,Simples,Reta,Năo,3,0,0,1,1,1,1,3,-18.48261,-39.92379,SPRF-ES,DEL04-ES,UOP02-DEL04-ES
571804.0,2024-01-01,segunda-feira,2025-04-14T04:50:00.000+0000,PI,343,185,PIRIPIRI,Manobra de mudança de faixa,Colisăo frontal,Com Vítimas Fatais,Amanhecer,Decrescente,Céu Claro,Simples,Reta,Sim,2,1,0,0,1,0,0,2,-4.29603281,-41.76732659,SPRF-PI,DEL02-PI,UOP01-DEL02-PI
571806.0,2024-01-01,segunda-feira,2025-04-14T04:30:00.000+0000,BA,116,578,BREJOES,Ingestăo de álcool pelo condutor,Colisăo frontal,Com Vítimas Fatais,Plena Noite,Decrescente,Céu Claro,Simples,Curva,Năo,3,1,0,0,1,2,0,4,-13.07158302,-39.9611107,SPRF-BA,DEL03-BA,UOP02-DEL03-BA
571818.0,2024-01-01,segunda-feira,2025-04-14T06:30:00.000+0000,SE,101,18,MALHADA DOS BOIS,Reaçăo tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Amanhecer,Crescente,Céu Claro,Dupla,Declive;Reta,Năo,2,0,0,1,0,2,1,3,-10.35601949,-36.90552235,SPRF-SE,DEL02-SE,UOP02-DEL02-SE
571838.0,2024-01-01,segunda-feira,2025-04-14T05:00:00.000+0000,MT,364,240,RONDONOPOLIS,Condutor deixou de manter distância do veículo da frente,Colisăo traseira,Sem Vítimas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Năo,3,0,0,0,2,1,0,3,-16.17914141,-54.78905337,SPRF-MT,DEL02-MT,UOP01-DEL02-MT


In [0]:
%sql
SELECT * FROM datatran_bronze.estados LIMIT 5;

Estado,Sigla,Cod
Acre,AC,12
Alagoas,AL,27
Amapá,AP,16
Amazonas,AM,13
Bahia,BA,29


SILVER LAYER

In [0]:
%sql
DROP DATABASE IF EXISTS datatran_silver CASCADE;
CREATE DATABASE datatran_silver;

In [0]:
#Eliminação de colunas que não serão usadas nas análises
colunasSemUso = ["km",
                 "municipio",
                 "fase_dia",
                "sentido_via",
                "tipo_pista",
                "tracado_via",
                "uso_solo",
                "pessoas",
                "feridos",
                "latitude",
                "longitude",
                "regional",
                "delegacia",
                "uop"]
for coluna in colunasSemUso:
    df_acidentes = df_acidentes.drop(coluna)

In [0]:
#Coluna classificacao_acidente 
df_acidentes = df_acidentes.filter(df_acidentes["classificacao_acidente"] != "NA")

#Coluna br
df_acidentes = df_acidentes.filter(df_acidentes["br"] != 0)

In [0]:
#Coluna data
df_acidentes = df_acidentes.withColumn("data_inversa", date_format("data_inversa", "dd-MM-yyyy"))
df_acidentes = df_acidentes.withColumnRenamed("data_inversa", "data")

#Coluna horario
df_acidentes = df_acidentes.withColumn("horario", date_format(to_timestamp("horario", "HH:mm"), "HH:mm"))


In [0]:
#Carga da tabela acidentes
df_acidentes.write.mode("overwrite").saveAsTable("datatran_silver.acidentes")

#Carga da tabela estados
df_estados.write.mode("overwrite").saveAsTable("datatran_silver.estados")

In [0]:
%sql
SELECT * FROM datatran_silver.acidentes LIMIT 5;

id,data,dia_semana,horario,uf,br,causa_acidente,tipo_acidente,classificacao_acidente,condicao_metereologica,mortos,feridos_leves,feridos_graves,ilesos,ignorados,veiculos
571804.0,01-01-2024,segunda-feira,04:50,PI,343,Manobra de mudança de faixa,Colisăo frontal,Com Vítimas Fatais,Céu Claro,1,0,0,1,0,2
571806.0,01-01-2024,segunda-feira,04:30,BA,116,Ingestăo de álcool pelo condutor,Colisăo frontal,Com Vítimas Fatais,Céu Claro,1,0,0,1,2,4
571818.0,01-01-2024,segunda-feira,06:30,SE,101,Reaçăo tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Céu Claro,0,0,1,0,2,3
571838.0,01-01-2024,segunda-feira,05:00,MT,364,Condutor deixou de manter distância do veículo da frente,Colisăo traseira,Sem Vítimas,Céu Claro,0,0,0,2,1,3
571855.0,01-01-2024,segunda-feira,11:50,MG,251,Velocidade Incompatível,Colisăo traseira,Com Vítimas Feridas,Chuva,0,1,0,1,1,3


In [0]:
%sql
SELECT * FROM datatran_silver.estados LIMIT 5;

Estado,Sigla,Cod
Acre,AC,12
Alagoas,AL,27
Amapá,AP,16
Amazonas,AM,13
Bahia,BA,29


GOLD LAYER

In [0]:
%sql
DROP DATABASE IF EXISTS datatran_gold CASCADE;
CREATE DATABASE datatran_gold;

In [0]:
"""
:param df: DataFrame
    DataFrame de origem dos dados das tabelas dimensão dias, causas, tipos, classificacoes e condicoes
:param coluna: Str
    identificação da coluna específica para cada tabela
:param nome_tabela: Str
    nome de cada tabela dimensão
"""
def buildDim(df, coluna, nome_tabela):
    #Separa a coluna necessária para a nova tabela
    df_dim = df.select(coluna).distinct()
    
    #Gera a coluna de código
    window = Window.orderBy(coluna)
    df_dim = df_dim.withColumn(f"cod_{coluna}", row_number().over(window))

    #Carga de dados
    df_dim.write.mode("overwrite").saveAsTable(f"datatran_gold.{nome_tabela}")
    return df_dim



In [0]:
#Conjunto das tabelas dimensão que serão criadas e carregadas em lote
colunasDim = [('dia_semana', 'dias'),      
              ('causa_acidente','causas'),
              ('tipo_acidente','tipos'),
              ('classificacao_acidente','classicacoes'),
              ('condicao_metereologica','condicoes')]

#Chamada da função de criação e carregamento
dfs_dimensoes = []
for coluna, nome_tabla in colunasDim:
    dfs_dimensoes.append(buildDim(df_acidentes, coluna, nome_tabla))



In [0]:
#Seleção das colunas para a tabela dimensão registros
df_registros = df_acidentes.select('id',
                                'data',
                                'horario',
                                'br',
                                'mortos',
                                'feridos_leves',
                                'feridos_graves',
                                'ilesos',
                                'ignorados',
                                'veiculos')

#Carga de dados
df_registros.write.mode("overwrite").saveAsTable(f"datatran_gold.registros")

In [0]:
#Carga da tabela estados
df_estados.write.mode("overwrite").saveAsTable("datatran_gold.estados")

In [0]:
df_dias, df_causas, df_tipos, df_classificacoes, df_condicoes = dfs_dimensoes
df_estados = df_estados.withColumnRenamed("Sigla", "uf")

In [0]:

df_ocorrencias = df_acidentes.join(df_dias, on="dia_semana", how="right")
df_ocorrencias = df_ocorrencias.drop("dia_semana")

df_ocorrencias = df_ocorrencias.join(df_estados, on="uf", how="right")
df_ocorrencias = df_ocorrencias.drop("uf")
df_ocorrencias = df_ocorrencias.drop("Estado")

df_ocorrencias = df_ocorrencias.join(df_causas, on="causa_acidente", how="right")
df_ocorrencias = df_ocorrencias.drop("causa_acidente")


df_ocorrencias = df_ocorrencias.join(df_tipos, on="tipo_acidente", how="right")
df_ocorrencias = df_ocorrencias.drop("tipo_acidente")

df_ocorrencias = df_ocorrencias.join(df_classificacoes, on="classificacao_acidente", how="right")
df_ocorrencias = df_ocorrencias.drop("classificacao_acidente")

df_ocorrencias = df_ocorrencias.join(df_condicoes, on="condicao_metereologica", how="right")
df_ocorrencias = df_ocorrencias.drop("condicao_metereologica")


df_ocorrencias.write.mode("overwrite").saveAsTable("datatran_gold.ocorrencias")

In [0]:
%sql
SELECT * FROM datatran_gold.ocorrencias LIMIT 5;

id,data,horario,br,mortos,feridos_leves,feridos_graves,ilesos,ignorados,veiculos,cod_dia_semana,Cod,cod_causa_acidente,cod_tipo_acidente,cod_classificacao_acidente,cod_condicao_metereologica
659177.0,24-11-2024,21:30,222,1,0,0,0,0,1,1,23,9,4,1,10
652449.0,31-12-2024,21:40,277,0,0,1,0,0,1,7,41,9,4,2,10
650224.0,21-12-2024,15:00,153,0,2,0,0,0,1,6,35,68,4,2,10
649695.0,19-12-2024,22:04,116,0,1,0,1,0,1,3,35,24,17,2,10
648194.0,13-12-2024,16:55,101,0,1,0,2,0,3,5,26,62,6,2,10


In [0]:
%sql
SELECT * FROM datatran_gold.dias;

dia_semana,cod_dia_semana
domingo,1
quarta-feira,2
quinta-feira,3
segunda-feira,4
sexta-feira,5
sábado,6
terça-feira,7


In [0]:
%sql
SELECT * FROM datatran_gold.causas;

causa_acidente,cod_causa_acidente
Acessar a via sem observar a presença dos outros veículos,1
Acesso irregular,2
Acostamento em desnível,3
Acumulo de areia ou detritos sobre o pavimento,4
Acumulo de água sobre o pavimento,5
Acumulo de óleo sobre o pavimento,6
Afundamento ou ondulaçăo no pavimento,7
Animais na Pista,8
Ausęncia de reaçăo do condutor,9
Ausęncia de sinalizaçăo,10


In [0]:
%sql
SELECT * FROM datatran_gold.tipos;

tipo_acidente,cod_tipo_acidente
Atropelamento de Animal,1
Atropelamento de Pedestre,2
Capotamento,3
Colisăo com objeto,4
Colisăo frontal,5
Colisăo lateral mesmo sentido,6
Colisăo lateral sentido oposto,7
Colisăo transversal,8
Colisăo traseira,9
Derramamento de carga,10


In [0]:
%sql
SELECT * FROM datatran_gold.classicacoes;

classificacao_acidente,cod_classificacao_acidente
Com Vítimas Fatais,1
Com Vítimas Feridas,2
Sem Vítimas,3


In [0]:
%sql
SELECT * FROM datatran_gold.condicoes;

condicao_metereologica,cod_condicao_metereologica
Chuva,1
Céu Claro,2
Garoa/Chuvisco,3
Granizo,4
Ignorado,5
Neve,6
Nevoeiro/Neblina,7
Nublado,8
Sol,9
Vento,10


In [0]:
%sql
SELECT * FROM datatran_gold.registros LIMIT 5;

id,data,horario,br,mortos,feridos_leves,feridos_graves,ilesos,ignorados,veiculos
571804.0,01-01-2024,04:50,343,1,0,0,1,0,2
571806.0,01-01-2024,04:30,116,1,0,0,1,2,4
571818.0,01-01-2024,06:30,101,0,0,1,0,2,3
571838.0,01-01-2024,05:00,364,0,0,0,2,1,3
571855.0,01-01-2024,11:50,251,0,1,0,1,1,3


In [0]:
%sql
SELECT * FROM datatran_gold.estados LIMIT 5;

Estado,Sigla,Cod
Acre,AC,12
Alagoas,AL,27
Amapá,AP,16
Amazonas,AM,13
Bahia,BA,29


### 6 - DIAGRAMA MER CAMADA GOLD

![](https://raw.githubusercontent.com/MdotSouza/PUCRio/main/EngDados_MVP/resources/MER.png)

### 7 - ANÁLISES

In [0]:
%sql
SELECT br, SUM(mortos)
FROM datatran_gold.ocorrencias
GROUP BY br
ORDER BY SUM(mortos) DESC
LIMIT 5;

br,sum(mortos)
116,821
101,732
153,270
163,240
316,221


In [0]:
%sql
SELECT d.dia_semana, SUM(o.mortos)
FROM datatran_gold.dias AS d
    INNER JOIN datatran_gold.ocorrencias AS o
        ON d.cod_dia_semana = o.cod_dia_semana 
    INNER JOIN datatran_gold.registros AS r 
        ON o.id = r.id
GROUP BY d.dia_semana
ORDER BY SUM(o.mortos) DESC
LIMIT 5;

dia_semana,sum(mortos)
domingo,1231
sábado,1105
sexta-feira,893
segunda-feira,763
quinta-feira,755


In [0]:
%sql
SELECT t.tipo_acidente, COUNT(*)
FROM datatran_gold.tipos AS t
    INNER JOIN datatran_gold.ocorrencias AS o
        ON t.cod_tipo_acidente = o.cod_tipo_acidente
    INNER JOIN datatran_gold.condicoes AS c
        ON c.cod_condicao_metereologica = o.cod_condicao_metereologica
    WHERE c.condicao_metereologica = 'Chuva'
GROUP BY t.tipo_acidente
ORDER BY COUNT(*) DESC
LIMIT 5;

tipo_acidente,count(1)
Saída de leito carroçável,2191
Colisăo traseira,939
Colisăo com objeto,855
Colisăo frontal,611
Tombamento,600


### 8 - AUTOAVALIAÇÃO
Este trabalho me proporcionou um aprendizado prático valioso sobre o uso do Spark e do Databricks para construção de pipelines de dados. Pude aplicar conceitos de tratamento, transformação e junção de dados, além de realizar análises usando Spark SQL. Enfrentei alguns desafios, especialmente na manipulação de DataFrames e estruturação do pipeline, mas consegui superá-los com prática e pesquisa. De modo geral, considero que evoluí bastante na compreensão de ferramentas de Big Data e me sinto mais preparado para projetos similares no futuro.